
# Set D — Passive Membrane as a Basic Electrical Circuit (LEGO–Colab)

**Final (Option 1 style)** — 2026-03-09  \
**Author:** Satish Nair

This notebook focuses **only** on the passive membrane: a leak conductance and a membrane capacitance.  
Run the **Starter** once, then execute D1…D10 in order. Each section includes a short **systems‑theoretic callout** and a **Try with Copilot** prompt.

---
## Table of Contents
- [🔰 Starter](#starter)
- [D1 — Nernst potential → leak reversal `e_pas`](#d1)
- [D2 — Step response & time constant τ](#d2)
- [D3 — Brief pulse (impulse‑like) kernel](#d3)
- [D4 — Sinusoidal input: amplitude & phase (frequency response)](#d4)
- [D5 — Input resistance R_in vs `g_pas`](#d5)
- [D6 — Spatial attenuation with a passive dendrite](#d6)
- [D7 — Estimate the space constant λ](#d7)
- [D8 — Sealed‑end effects vs longer cable](#d8)
- [D9 — Temporal & spatial summation](#d9)
- [D10 — Playground: sweep τ, R_in, attenuation](#d10)
- [Reflection](#reflection)
---


### 🔰 Starter (run once at the very top) <a id='starter'></a>

In [ ]:

# --- Set D Starter: Install NEURON and create a passive single-compartment cell ---
!pip -q install neuron==8.2.4
try:
    from neuron import h, gui
except Exception:
    from neuron import h
from neuron.units import ms, mV
import matplotlib.pyplot as plt
import numpy as np

h.load_file("stdrun.hoc")

# Create a spherical soma (single compartment)
soma = h.Section(name='soma')
soma.L = 20    # µm
soma.diam = 20 # µm
soma.Ra = 100  # ohm·cm

# Insert passive leak only
soma.insert('pas')
soma.g_pas = 0.0002     # S/cm^2 (leak conductance density)
soma.e_pas = -65        # mV (leak reversal)
soma.cm    = 1.0        # uF/cm^2

# Recorders
t = h.Vector().record(h._ref_t)
v = h.Vector().record(soma(0.5)._ref_v)

# Utility to run and plot
def run(sim_ms=60, v_init=-65, title="", figsize=(7,3.3)):
    h.finitialize(v_init*mV)
    h.continuerun(sim_ms*ms)
    plt.figure(figsize=figsize)
    plt.plot(t, v, 'k')
    plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title(title)
    plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()

print('Set D starter ready — passive membrane initialized.')


## D1 — Re‑emphasize Nernst Potential (biology → parameter `e_pas`) <a id='d1'></a>
Define a helper to compute Nernst for a monovalent ion and map it to leak reversal for demonstration.

In [ ]:

# Helper: Nernst potential (natural log) at temperature T_C (Celsius)
R = 8.31446261815324  # J/(mol·K)
F = 96485.33212       # C/mol

def nernst_mV(cin_mM, cout_mM, z=1, T_C=37.0):
    T_K = 273.15 + T_C
    return (1000.0*R*T_K/(z*F))*np.log(cout_mM/cin_mM)  # mV (natural log)

for ion, params in {
    'K+':  {'cin':140, 'cout':5,  'z':1},
    'Na+': {'cin':12,  'cout':145,'z':1},
    'Cl-': {'cin':5,   'cout':120,'z':-1}
}.items():
    E = nernst_mV(params['cin'], params['cout'], params['z'], 37.0)
    print(f"E_{ion} ≈ {E:.1f} mV at 37°C")

# Demonstration: set leak reversal near potassium-like value
soma.e_pas = float(nernst_mV(140,5,1,37.0))
print('Set e_pas to ~E_K for demonstration:', soma.e_pas)

# Small negative pulse to see resting level near E_K
stim = h.IClamp(soma(0.5)); stim.delay=5; stim.dur=50; stim.amp=-0.03
run(sim_ms=70, title='D1: Passive cell with e_pas near E_K (Nernst review)')

# Restore classroom default for next steps
soma.e_pas = -65; stim.amp = 0.0


> **Systems callout — Rest potential as fixed point**  
- In a purely passive cell, the equilibrium approaches the leak reversal `e_pas`.
- Setting `e_pas` near an ion’s Nernst potential demonstrates biology→parameter mapping.

**Try with Copilot**  
> Using your lab’s typical [K⁺]_in and [K⁺]_out at 32–34°C, estimate `E_K` and discuss how temperature affects `E_K`.

## D2 — First‑order response (step current) and time constant τ <a id='d2'></a>

In [ ]:

# Step stimulus (hyperpolarizing to stay passive)
stim.delay = 5; stim.dur = 60; stim.amp = -0.05
h.finitialize(-65*mV); h.continuerun(80*ms)

# Estimate steady-state change and tau from rising edge
t_np = np.array(t); v_np = np.array(v)
ss_mask = (t_np>60) & (t_np<63)
v_ss = v_np[ss_mask].mean()
# target is 63.2% of total change
v0_idx = np.where(t_np>=5)[0][0]
v0 = v_np[v0_idx]
vtgt = v0 + 0.632*(v_ss - v0)
idx = np.where(v_np <= vtgt)[0] if stim.amp<0 else np.where(v_np >= vtgt)[0]

tau_est = (t_np[idx[0]] - 5.0) if len(idx)>0 else float('nan')
print(f"Estimated τ ≈ {tau_est:.2f} ms, steady ΔV ≈ {v_ss - v0:.2f} mV")

plt.figure(figsize=(7,3.3))
plt.plot(t_np, v_np, 'k'); plt.axvline(5, color='gray', ls='--', lw=1)
plt.axhline(v_ss, color='C1', ls=':')
plt.axhline(vtgt, color='C2', ls=':')
plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title('D2: Step response and τ estimate')
plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — τ ≈ C_total / g_total**  
- For small perturbations, passive dynamics are first‑order with time constant τ.
- Bigger `cm` → larger τ (slower); bigger `g_pas` → smaller τ (faster). 

**Try with Copilot**  
> From your trace, compute τ using both 63% and a log‑linear fit; compare values and discuss noise sensitivity.

## D3 — Brief pulse “impulse” and passive kernel <a id='d3'></a>

In [ ]:

stim.amp = -0.2; stim.dur = 1.0; stim.delay = 5
run(sim_ms=30, title='D3: Brief pulse response (impulse-like)')
# Restore
stim.amp=0.0


> **Systems callout — Impulse response**  
- The passive membrane acts as a first‑order low‑pass; the impulse response is an exponential kernel.
- Area under V(t) relates to effective RC; shape governed by τ.

**Try with Copilot**  
> Estimate τ from the envelope of the impulse response; compare with D2’s τ.

## D4 — Sinusoidal current: amplitude and phase (frequency response) <a id='d4'></a>

In [ ]:

# Drive with a sine using Vector.play to IClamp
def sine_run(freq_hz=10, amp_nA=0.05, dur_ms=400):
    stim.amp = 0.0
    ic = h.IClamp(soma(0.5)); ic.delay=0; ic.dur=1e9; ic.amp=0.0
    # time vector in ms
    tt = np.arange(0, dur_ms, 0.025)
    ii = amp_nA*np.sin(2*np.pi*freq_hz*(tt/1000.0))
    ivec = h.Vector(ii); tvec = h.Vector(tt)
    ivec.play(ic._ref_amp, tvec, 1)
    h.finitialize(-65*mV); h.continuerun(dur_ms*ms)
    # compute amplitude (peak-to-peak/2) after a transient
    t_np = np.array(t); v_np = np.array(v)
    mask = t_np > (dur_ms*0.5)
    v_amp = (v_np[mask].max() - v_np[mask].min())/2
    plt.figure(figsize=(7,3.3))
    plt.plot(t_np, v_np, 'k')
    plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title(f'D4: Sine drive {freq_hz} Hz, Vm amp ≈ {v_amp:.2f} mV')
    plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()

for f in [2, 10, 50]:
    sine_run(freq_hz=f, amp_nA=0.05, dur_ms=400)


> **Systems callout — Frequency response of an RC**  
- Low frequencies pass with larger amplitude; high frequencies are attenuated and phase‑lagged.
- Cutoff ~ 1/(2πτ): use D2’s τ to predict attenuation at 10–50 Hz.

**Try with Copilot**  
> Predict the gain |V/I| at 2, 10, and 50 Hz from τ; compare with measured amplitudes.

## D5 — Input resistance R_in and its relation to leak conductance <a id='d5'></a>

In [ ]:

# Measure Rin ≈ ΔV/ΔI for several g_pas values
def measure_rin(gpas=0.0002, amp=-0.03):
    soma.g_pas = gpas
    stim.delay=5; stim.dur=40; stim.amp=amp
    h.finitialize(-65*mV); h.continuerun(60*ms)
    t_np = np.array(t); v_np = np.array(v)
    ss = (t_np>40) & (t_np<45)
    dv = v_np[ss].mean() - (-65)
    Rin = abs(dv/amp)  # mV/nA = MΩ
    return Rin, dv

vals = []
for g in [0.0001, 0.0002, 0.0005, 0.001]:
    Rin, dv = measure_rin(g)
    vals.append((g, Rin))
    print(f"g_pas={g:.5f} S/cm²  =>  R_in≈{Rin:.1f} MΩ (ΔV≈{dv:.2f} mV)")

soma.g_pas = 0.0002; stim.amp=0.0


> **Systems callout — Ohm’s law at steady state**  
- R_in ≈ ΔV/ΔI reflects the *total* leak conductance.
- Doubling `g_pas` halves `R_in` (approximately, for a single compartment). 

**Try with Copilot**  
> Fit Rin vs g_pas to a 1/g line; report deviations and hypothesize sources (geometry, discretization).

## D6 — Spatial attenuation with a passive dendrite (space constant λ) <a id='d6'></a>

In [ ]:

# Build dendrite and record at several locations (0..1)
try:
    dend = dend
except NameError:
    dend = h.Section(name='dend')
    dend.L = 800; dend.diam = 1.5; dend.Ra = 100
    dend.insert('pas'); dend.g_pas = soma.g_pas; dend.e_pas = soma.e_pas
    dend.connect(soma(1.0))

sites = [0.0, 0.25, 0.5, 0.75, 1.0]
recs  = [h.Vector().record(dend(x)._ref_v) for x in sites]

stim_d = h.IClamp(dend(1.0)); stim_d.delay=5; stim_d.dur=40; stim_d.amp=-0.05
h.finitialize(-65*mV); h.continuerun(70*ms)

plt.figure(figsize=(7,3.3))
for x,vec in zip(sites,recs):
    plt.plot(t, vec, label=f'dend({x:.2f})')
plt.plot(t, v, label='soma(0.5)', lw=2, color='k')
plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title('D6: Passive attenuation along dendrite')
plt.legend(); plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — Space‑dependent filtering**  
- Voltage attenuates with distance along a passive cable; distal inputs are smaller and slower at soma.
- λ depends on axial resistance, membrane resistance, and diameter. 

**Try with Copilot**  
> Predict attenuation at dend(0.75) vs dend(0.25) given λ and compare to your trace.

## D7 — Estimate the space constant λ from steady profile <a id='d7'></a>

In [ ]:

# Record steady voltages at multiple locations during a long distal pulse
sites2 = [0.0, 0.25, 0.5, 0.75, 1.0]
recs2  = [h.Vector().record(dend(x)._ref_v) for x in sites2]

stim_d.delay=5; stim_d.dur=200; stim_d.amp=-0.05
h.finitialize(-65*mV); h.continuerun(220*ms)

# Compute steady values from the tail of the traces
steady = []
for vec in recs2:
    arr = np.array(vec)
    steady.append(arr[-50:].mean())

V_soma = np.array(v)[-50:].mean()
Vrest = -65.0
xs = np.array(sites2) * 800.0  # µm (same L as D6)
mag = np.abs(np.array(steady) - Vrest) + 1e-6

# Fit log-linear: ln(|V(x)-V_rest|) ~ -x/lambda + c
coef = np.polyfit(xs, np.log(mag), 1)
lambda_um = -1.0/coef[0]
print(f"Estimated λ ≈ {lambda_um:.1f} µm (didactic fit)")

plt.figure(figsize=(6,3.6))
plt.plot(xs, mag, 'o', label='data')
plt.plot(xs, np.exp(coef[1] + coef[0]*xs), '-', label='exp fit')
plt.xlabel('Distance (µm)'); plt.ylabel('|V - V_rest| (mV)'); plt.title('D7: λ estimate from steady attenuation')
plt.legend(); plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — Estimating λ**  
- For a passive cable, |V(x)−V_rest| often decays ~exp(−x/λ) under steady drive.
- Diameter ↑ and membrane resistance ↑ typically increase λ (less attenuation). 

**Try with Copilot**  
> Repeat the fit after doubling dendrite diameter and report the new λ; explain the change.

## D8 — Sealed‑end effects vs longer cable <a id='d8'></a>

In [ ]:

# Compare sealed-end dendrites of varying length
def run_dend(L_um=400):
    d = h.Section(name=f'dend_{L_um}')
    d.L = L_um; d.diam = 1.5; d.Ra = 100
    d.insert('pas'); d.g_pas=soma.g_pas; d.e_pas=soma.e_pas
    d.connect(soma(1.0))
    vtip = h.Vector().record(d(1.0)._ref_v)
    stimx = h.IClamp(d(1.0)); stimx.delay=5; stimx.dur=40; stimx.amp=-0.05
    h.finitialize(-65*mV); h.continuerun(70*ms)
    return np.array(t), np.array(v), np.array(vtip)

for L in [300, 800, 1500]:
    tt, vsoma, vtip = run_dend(L)
    plt.figure(figsize=(7,3.3))
    plt.plot(tt, vsoma, label='soma')
    plt.plot(tt, vtip, label=f'tip (L={L} µm)')
    plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title('D8: Sealed-end length effects')
    plt.legend(); plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — Boundary conditions**  
- Sealed ends reflect current; length changes attenuation and timing.
- As L increases, distal responses become smaller/slower at soma.

**Try with Copilot**  
> Predict how attaching a second branch at dend(0.5) changes the tip response and why.

## D9 — Temporal and spatial summation in a passive cell <a id='d9'></a>

In [ ]:

# Temporal summation at soma
stim = h.IClamp(soma(0.5)); stim.delay=5; stim.dur=3; stim.amp=0.05
# play a train via Vector.play
tt = np.arange(0, 120, 0.1)
ii = np.zeros_like(tt)
for k in range(10):
    onset = 10 + k*5  # ms
    ii[(tt>=onset) & (tt<onset+1)] = 0.05
ivec = h.Vector(ii); tvec = h.Vector(tt)
ivec.play(stim._ref_amp, tvec, 1)
h.finitialize(-65*mV); h.continuerun(120*ms)
plt.figure(figsize=(7,3.3))
plt.plot(t, v, 'k'); plt.title('D9: Temporal summation (soma)')
plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()

# Spatial summation: two distal sites on dendrite
ic1 = h.IClamp(dend(0.8)); ic1.delay=10; ic1.dur=5; ic1.amp=-0.05
ic2 = h.IClamp(dend(0.2)); ic2.delay=10; ic2.dur=5; ic2.amp=-0.05
h.finitialize(-65*mV); h.continuerun(40*ms)
plt.figure(figsize=(7,3.3))
plt.plot(t, v, 'k'); plt.title('D9: Spatial summation (dend(0.2)+dend(0.8))')
plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — Linear integration (passive)**  
- Temporal summation depends on τ; closer pulses sum more.
- Spatial summation depends on λ and the electrotonic distances of input sites.

**Try with Copilot**  
> Increase pulse rate until summation plateaus; explain using τ and steady‑state limits.

## D10 — Playground: parameter sweep for τ, R_in, and attenuation <a id='d10'></a>

In [ ]:

# Sweep membrane parameters (cm, g_pas, dend diameter) and observe effects on τ and Rin
def passive_summary(cm=1.0, gpas=0.0002, diam=1.5):
    soma.cm = cm; soma.g_pas = gpas
    dend.diam = diam
    # τ from D2-like pulse
    stim.delay=5; stim.dur=40; stim.amp=-0.05
    h.finitialize(-65*mV); h.continuerun(60*ms)
    t_np = np.array(t); v_np = np.array(v)
    ss = (t_np>40) & (t_np<45)
    v_ss = v_np[ss].mean()
    v0 = v_np[np.where(t_np>=5)[0][0]]
    vtgt = v0 + 0.632*(v_ss - v0)
    idx = np.where(v_np <= vtgt)[0]
    tau_est = (t_np[idx[0]] - 5.0) if len(idx)>0 else np.nan
    Rin = abs((v_ss - v0)/(-0.05))
    return tau_est, Rin

settings = [
    (1.0, 0.0002, 1.5),
    (2.0, 0.0002, 1.5),
    (1.0, 0.0005, 1.5),
    (1.0, 0.0002, 3.0),
]
for cm,gpas,diam in settings:
    tau_est, Rin = passive_summary(cm, gpas, diam)
    print(f"cm={cm}, g_pas={gpas}, dend.diam={diam}  =>  tau≈{tau_est:.2f} ms, R_in≈{Rin:.1f} MΩ")

# Restore defaults
soma.cm=1.0; soma.g_pas=0.0002; dend.diam=1.5; stim.amp=0.0


> **Systems callout — Parameter sensitivities**  
- τ scales with C and inverse with total leak; Rin inversely with leak.
- Diameter influences axial resistance and λ, altering spatial integration.

**Try with Copilot**  
> Design a parameter set that yields τ≈15 ms and Rin≈150 MΩ; report cm and g_pas choices.

## Reflection <a id='reflection'></a>
- How does the passive RC viewpoint help predict responses to arbitrary inputs (superposition)?  
- Which experimental measurements most robustly estimate τ and Rin in noisy recordings, and why?  
- How do λ and τ jointly shape temporal vs spatial summation in dendrites?